# Part 6: Exporting Data

Once you have queried your data, you often need to export it for downstream analysis (R, Excel, pandas, etc.). In this notebook we cover:

- Exporting query results to **CSV**
- Exporting to **TSV**
- Using `pandas` to write Excel files
- Loading an exported file back in to verify round-trip integrity

> **Prerequisite:** Run `01_setup.ipynb` first.

In [ ]:
import sqlite3
import csv
import pandas as pd
from pathlib import Path

con = sqlite3.connect("gene_expression.db")

def query(sql, params=()):
    return pd.read_sql(sql, con, params=params)

print("Connected.")

---
## 1. Export to CSV using Python's `csv` module

In [ ]:
sql = """
SELECT g.symbol, g.chromosome, s.name AS sample, s.condition, s.tissue,
       e.tpm, e.raw_counts
FROM   expression e
JOIN   genes      g ON g.gene_id   = e.gene_id
JOIN   samples    s ON s.sample_id = e.sample_id
ORDER BY g.symbol, s.name
"""

cur = con.execute(sql)
columns = [d[0] for d in cur.description]
rows = cur.fetchall()

out_csv = Path("expression_export.csv")
with open(out_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(columns)   # header
    writer.writerows(rows)

print(f"Exported {len(rows)} rows to {out_csv}")

---
## 2. Export to TSV

In [ ]:
out_tsv = Path("expression_export.tsv")
with open(out_tsv, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(columns)
    writer.writerows(rows)

print(f"Exported {len(rows)} rows to {out_tsv}")

---
## 3. Export using pandas — simpler and more flexible

In [ ]:
df = query(sql)
print(f"DataFrame shape: {df.shape}")
df.head()

In [ ]:
# CSV
df.to_csv("expression_pandas.csv", index=False)
print("Saved expression_pandas.csv")

# TSV
df.to_csv("expression_pandas.tsv", sep="\t", index=False)
print("Saved expression_pandas.tsv")

---
## 4. Export a pivot table — genes as rows, samples as columns

In [ ]:
df_pivot = df.pivot(index="symbol", columns="sample", values="tpm")
df_pivot

In [ ]:
df_pivot.to_csv("expression_matrix.csv")
print("Saved expression_matrix.csv (genes x samples)")

---
## 5. Verify round-trip — read back and compare

In [ ]:
df_reload = pd.read_csv("expression_pandas.csv")

print(f"Original shape: {df.shape}")
print(f"Reloaded shape: {df_reload.shape}")
print(f"\nColumns match: {list(df.columns) == list(df_reload.columns)}")

# Compare TPM values (rounded to avoid float precision issues)
match = (df["tpm"].round(2) == df_reload["tpm"].round(2)).all()
print(f"TPM values match: {match}")

---
## 6. List all exported files

In [ ]:
for f in sorted(Path(".").glob("expression_*")):
    size = f.stat().st_size
    print(f"  {f.name:<35} {size:>8,} bytes")

---
## Exercises

**Q1.** Export only the tumor samples to a CSV called `tumor_expression.csv`.

In [ ]:
# Your code here


**Q2.** Create a pivot table with **samples as rows** and **genes as columns** (the transpose of what we did above). Export it to `expression_matrix_transposed.csv`.

In [ ]:
# Your code here


**Q3.** Export the gene metadata table (`genes`) to a TSV file called `genes.tsv`.

In [ ]:
# Your code here


In [ ]:
con.close()
print("Done. All exports complete.")